# Daily Challenge – Week 5, Day 2: Breast Cancer Prediction
Models compared: Logistic Regression, K-Nearest Neighbours, Random Forest and SVM.

**Dataset:** Breast Cancer Wisconsin (Diagnostic). If you downloaded the Kaggle file (`data.csv`), put it next to this notebook and it will be used; otherwise the notebook loads the **same dataset** that is built into scikit-learn (569 tumours, 30 measurements), so no Kaggle login is needed.

## 1. Exploratory Data Analysis

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

if os.path.exists("data.csv"):
    df = pd.read_csv("data.csv")                    # Kaggle version
else:
    raw = load_breast_cancer(as_frame=True)         # same data, built into scikit-learn
    df = raw.frame.copy()
    df.columns = [c.replace(" ", "_") for c in df.columns]
    # scikit-learn codes the target as 0 = malignant, 1 = benign; recreate Kaggle's M/B column
    df.insert(0, "diagnosis", np.where(df["target"] == 0, "M", "B"))
    df = df.drop(columns="target")
    df.insert(0, "id", range(1, len(df) + 1))      # Kaggle has an id column too

print(df.shape)
df.head()

In [ ]:
# Missing values
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else "No missing values")

# Drop unnecessary columns: the id says nothing about the tumour,
# and the Kaggle file has an empty extra column called "Unnamed: 32"
df = df.drop(columns=[c for c in ["id", "Unnamed: 32"] if c in df.columns])
print("Shape after dropping:", df.shape)

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="diagnosis", hue="diagnosis", palette="magma", legend=False)
plt.title("Diagnosis counts (M = malignant, B = benign)")
plt.show()

## 2. Data Preprocessing, Building Models and Evaluation

In [ ]:
# Counts of unique values in 'diagnosis'
print(df["diagnosis"].value_counts())

# Map categorical values to numbers: M (malignant) = 1, B (benign) = 0
df["diagnosis"] = df["diagnosis"].map({"M": 1, "B": 0})
df["diagnosis"].value_counts()

In [ ]:
X = df.drop(columns="diagnosis")
y = df["diagnosis"]

# Stratified split keeps the same share of malignant tumours in train and test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

# LogReg, KNN and SVM are sensitive to feature scales (e.g. area ~1000 vs. smoothness ~0.1), so we standardize.
# The scaler is fitted on the training data only.
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)
print("Train:", X_train.shape, "Test:", X_test.shape)

In [ ]:
results = {}

# Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
log_reg.fit(X_train_s, y_train)
results["Logistic Regression"] = log_reg.predict(X_test_s)
print("Logistic Regression accuracy:", round(accuracy_score(y_test, results["Logistic Regression"]), 4))

# K-Nearest Neighbours
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_s, y_train)
results["KNN"] = knn.predict(X_test_s)
print("KNN accuracy:", round(accuracy_score(y_test, results["KNN"]), 4))

# Random Forest (trees don't need scaling)
rf = RandomForestClassifier(n_estimators=200, random_state=42)
rf.fit(X_train, y_train)
results["Random Forest"] = rf.predict(X_test)
print("Random Forest accuracy:", round(accuracy_score(y_test, results["Random Forest"]), 4))

# Support Vector Machine
svm = SVC(kernel="rbf", random_state=42)
svm.fit(X_train_s, y_train)
results["SVM"] = svm.predict(X_test_s)
print("SVM accuracy:", round(accuracy_score(y_test, results["SVM"]), 4))

In [ ]:
summary = pd.DataFrame({
    name: {"accuracy": accuracy_score(y_test, pred),
           "recall (malignant caught)": recall_score(y_test, pred),
           "missed malignant": confusion_matrix(y_test, pred)[1, 0]}
    for name, pred in results.items()
}).T.sort_values(["accuracy", "recall (malignant caught)"], ascending=False)
summary.round(4)

In [ ]:
summary["accuracy"].sort_values().plot(kind="barh", figsize=(7, 3.5), color=sns.color_palette("magma", 4))
plt.xlim(0.9, 1.0)
plt.title("Test accuracy by model")
plt.show()

## Which is the best model?
**The SVM is the best model here**, with **97.4% test accuracy**, followed by Logistic Regression and Random Forest (96.5%) and KNN (95.6%).

- All four models perform very well, because the 30 tumour measurements separate malignant and benign tumours clearly. The differences are only **1–2 test cases** out of 114, so with a different random split the ranking could change.
- In a medical setting, **recall for malignant tumours** matters more than accuracy: a missed cancer is much worse than a false alarm. SVM and Logistic Regression each missed **3 of 42 malignant tumours** (recall 92.9%), while Random Forest and KNN missed 4.
- **Logistic Regression** is almost as good as SVM and is easier to explain to doctors (each feature has a clear coefficient), so it would be a strong practical choice too.
- **Next steps** to choose more reliably: use cross-validation instead of one split, tune the hyperparameters (e.g. C and gamma for SVM, k for KNN), and consider lowering the decision threshold to catch more malignant cases.